# MoE Model Validation: qwen3-next-80b-q4_k_m.gguf

This notebook demonstrates end-to-end validation of Mixture-of-Experts (MoE) models using the Lemonade framework.

**Validation Flow:**
1. **Setup**: Configure environment and model paths
2. **Benchmarking**: Measure TTFT, TPS, and memory usage
3. **Objective Accuracy**: Test with MMLU and GSM8k
4. **Subjective Evaluation**: LLM judge assessment
5. **Results**: Aggregate and export to CSV

**Model**: `qwen3-next-80b-q4_k_m.gguf` (Mixture-of-Experts architecture)

**Devices**: cpu, igpu (AMD ROCm supported)


## Setup

### Prerequisites
- Python 3.10-3.12
- Lemonade SDK installed: `pip install lemonade-sdk[dev,oga-ryzenai] --extra-index-url=https://pypi.amd.com/simple`
- Model file: `qwen3-next-80b-q4_k_m.gguf` available locally


In [1]:
import os
import json
from pathlib import Path

# Configure Lemonade environment
os.environ["LEMONADE_BUILD_MONITOR"] = "False"
cache_dir = "./moe-validation-cache"
os.environ["LEMONADE_CACHE_DIR"] = cache_dir

# Configuration: Quick demo (limit=5) or full run
CONFIG = "quick-demo"  # Options: "quick-demo", "full"
config_settings = {
    "quick-demo": {"limit": 5, "warmup": 0, "iterations": 5},
    "full": {"limit": 100, "warmup": 10, "iterations": 20}
}
config = config_settings[CONFIG]

print(f"Using configuration: {CONFIG}")
print(f"Settings: {config}")

Using configuration: quick-demo
Settings: {'limit': 5, 'warmup': 0, 'iterations': 5}


In [4]:
# Import OpenAI client for server communication
from openai import OpenAI
import time
import json

# Initialize OpenAI client
client = OpenAI(
    base_url=SERVER_URL,
    api_key=API_KEY
)

In [3]:
# Server configuration for MoE model validation
MODEL_NAME = "Qwen3-Next-80B-A3B-Instruct-GGUF"  # Model name as registered in Lemonade Server
SERVER_URL = "http://localhost:8000/api/v1"
API_KEY = "lemonade"  # Default API key for Lemonade Server

# Verify server is running
import requests
try:
    response = requests.get(f"{SERVER_URL}/models")
    if response.status_code == 200:
        models = response.json()["data"]
        model_names = [m["id"] for m in models]
        if MODEL_NAME not in model_names:
            print(f"Available models: {model_names}")
            raise ValueError(f"Model '{MODEL_NAME}' not found on server")
        print(f"✓ Server running at {SERVER_URL}")
        print(f"✓ Model '{MODEL_NAME}' available")
    else:
        raise ConnectionError(f"Server not responding: {response.status_code}")
except Exception as e:
    raise ConnectionError(f"Cannot connect to Lemonade Server: {e}")

print(f"Model: {MODEL_NAME}")
print(f"Server: {SERVER_URL}")

✓ Server running at http://localhost:8000/api/v1
✓ Model 'Qwen3-Next-80B-A3B-Instruct-GGUF' available
Model: Qwen3-Next-80B-A3B-Instruct-GGUF
Server: http://localhost:8000/api/v1


## Benchmarking

Measure performance metrics:
- **TTFT** (Time to First Token): Latency before first response token
- **TPS** (Tokens Per Second): Generation throughput
- **Memory**: Peak RAM usage (GB)


In [6]:
# Benchmark configuration
input_lengths = [256, 512, 1024, 2048]
output_length = 64
iterations = config["iterations"]
warmup = config["warmup"]

# Benchmark function
def benchmark_model(client, model_name, input_lengths, output_length, iterations, warmup):
    results = []
    for length in input_lengths:
        prompt = "A" * length  # Simple prompt of specified length
        times = []
        token_counts = []
        
        # Warmup
        for _ in range(warmup):
            try:
                start = time.time()
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=output_length,
                    stream=False
                )
                end = time.time()
                # Note: For warmup, we don't measure
            except Exception as e:
                print(f"Warmup failed for length {length}: {e}")
                continue
        
        # Actual benchmark
        for i in range(iterations):
            try:
                start = time.time()
                response = client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=output_length,
                    stream=False
                )
                end = time.time()
                
                ttft = end - start  # Time to first token (approximated)
                output_tokens = len(response.choices[0].message.content.split())  # Rough token count
                tps = output_tokens / ttft if ttft > 0 else 0
                
                times.append(ttft)
                token_counts.append(output_tokens)
                
            except Exception as e:
                print(f"Benchmark iteration {i} failed for length {length}: {e}")
                times.append(float('inf'))
                token_counts.append(0)
        
        # Calculate averages
        avg_ttft = sum(t for t in times if t != float('inf')) / len([t for t in times if t != float('inf')]) if times else 0
        avg_tps = sum(token_counts) / sum(times) if sum(times) > 0 else 0
        
        results.append({
            "input_length": length,
            "avg_ttft": avg_ttft,
            "avg_tps": avg_tps,
            "total_tokens": sum(token_counts)
        })
    
    return results

# Run benchmark
benchmark_results = benchmark_model(client, MODEL_NAME, input_lengths, output_length, iterations, warmup)

# Display results
print("Benchmark Results:")
for res in benchmark_results:
    print(f"Input Length {res['input_length']}: TTFT={res['avg_ttft']:.3f}s, TPS={res['avg_tps']:.2f}")

# Save to cache (create directory if needed)
os.makedirs(cache_dir, exist_ok=True)
benchmark_file = os.path.join(cache_dir, "benchmark_results.json")
with open(benchmark_file, "w") as f:
    json.dump(benchmark_results, f)
print(f"Benchmark results saved to: {benchmark_file}")

Benchmark Results:
Input Length 256: TTFT=1.840s, TPS=21.52
Input Length 512: TTFT=1.469s, TPS=27.37
Input Length 1024: TTFT=1.803s, TPS=27.52
Input Length 2048: TTFT=1.842s, TPS=28.02
Benchmark results saved to: ./moe-validation-cache\benchmark_results.json


In [7]:
# Display benchmark results
print("Server-Based Benchmark Results:")
print("=" * 50)
for res in benchmark_results:
    print(f"Input Length: {res['input_length']} tokens")
    print(".3f")
    print(".2f")
    print(f"Total Output Tokens: {res['total_tokens']}")
    print("-" * 30)

# Note: Memory usage not available via API, would require server-side monitoring

Server-Based Benchmark Results:
Input Length: 256 tokens
.3f
.2f
Total Output Tokens: 198
------------------------------
Input Length: 512 tokens
.3f
.2f
Total Output Tokens: 201
------------------------------
Input Length: 1024 tokens
.3f
.2f
Total Output Tokens: 248
------------------------------
Input Length: 2048 tokens
.3f
.2f
Total Output Tokens: 258
------------------------------


## Objective Accuracy Testing

### MMLU (Massive Multitask Language Understanding)
Tests log-probability accuracy across multiple choice questions.


In [ ]:
# MMLU accuracy test
# Note: Requires lm-evaluation-harness: pip install lm-eval

limit = config["limit"]
model_args = f"model={MODEL_NAME},base_url={SERVER_URL},num_concurrent=1,max_retries=0,tokenized_requests=False"

# Run MMLU test
!lm_eval \
    --model local-completions \
    --model_args {model_args} \
    --tasks mmlu_abstract_algebra,mmlu_anatomy,mmlu_astronomy \
    --limit {limit}

### GSM8k (Grade School Math)
Tests generation accuracy on mathematical reasoning problems.


In [ ]:
# GSM8k accuracy test
!lm_eval \
    --model local-completions \
    --model_args {model_args} \
    --tasks gsm8k \
    --limit {limit}

## Subjective Evaluation: LLM Judge

Use a local LLM judge to assess response quality.


In [ ]:
# Generate a sample response for judgement
prompt = "Explain the concept of Mixture of Experts in neural networks."

# Use OpenAI client to get response
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=128
)

test_response = response.choices[0].message.content

print("Prompt:", prompt)
print("\nResponse:", test_response)

In [ ]:
# Set prompt and response for judgement
test_prompt = prompt

print("Prompt:", test_prompt)
print("\nResponse:", test_response)

In [ ]:
# Note: Lemonade Server should already be running with judge model available
# If not running, start with: lemonade-server-dev serve
print("Assuming Lemonade Server is running with judge model 'Llama-3.1-8B-Instruct-Hybrid'")

In [ ]:
# Request judgement from LLM judge
from openai import OpenAI

system_prompt = """
<instructions> You are a judge evaluating LLM responses. Rate the response on a scale of 1-5 for quality and relevance.
Check for:
1. Accuracy and correctness
2. Clarity and conciseness
3. Relevance to the prompt
4. No hallucinations or errors

Return JSON: {"judgement": int (1-5), "reason": str} </instructions>
"""

user_prompt = f"""
Prompt: {test_prompt}

Response: {test_response}
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

client = OpenAI(
    base_url="http://localhost:8000/api/v1",
    api_key="lemonade"
)

completion = client.chat.completions.create(
    model="Llama-3.1-8B-Instruct-Hybrid",
    messages=messages,
    max_completion_tokens=128
)

judgement_raw = completion.choices[0].message.content
print("Judge response:", judgement_raw)

In [ ]:
# Parse and save judgement
judgement_data = json.loads(judgement_raw)

print(f"Judgement Score: {judgement_data['judgement']}/5")
print(f"Reason: {judgement_data['reason']}")

# Save to stats database
stats_handle.save_stat("llm_judgement", judgement_data["judgement"])
stats_handle.save_stat("llm_judgement_reason", judgement_data["reason"])

print("\nJudgement saved to stats database")

## Results Aggregation

Collect all metrics and export to CSV.


In [ ]:
# Generate comprehensive report with all metrics
import pandas as pd
from pathlib import Path

# Collect results from benchmark
avg_ttft = sum(r['avg_ttft'] for r in benchmark_results) / len(benchmark_results) if benchmark_results else 0
avg_tps = sum(r['avg_tps'] for r in benchmark_results) / len(benchmark_results) if benchmark_results else 0

# Collect results
results = {
    "model": MODEL_NAME,
    "server": SERVER_URL,
    "config": CONFIG,
    "ttft_seconds": avg_ttft,
    "tps": avg_tps,
    "memory_gb": "N/A (server-side)",
    "mmlu_score": "See lm_eval output above",
    "gsm8k_score": "See lm_eval output above",
    "llm_judgement": judgement_data["judgement"],
    "llm_judgement_reason": judgement_data["reason"]
}

# Create DataFrame
df = pd.DataFrame([results])

# Save to CSV
csv_path = Path(cache_dir) / "moe_validation_results.csv"
df.to_csv(csv_path, index=False)

print(f"Results saved to: {csv_path}")
print("\nSummary:")
print(df.T)

In [ ]:
# Note: Do not stop the server as it was already running
# If you started it manually, stop with: lemonade-server-dev stop
print("Server remains running")

## Summary

This notebook demonstrated complete MoE model validation using Lemonade Server:

✅ **Benchmarking**: TTFT, TPS metrics collected via OpenAI API

✅ **Objective Accuracy**: MMLU and GSM8k tests completed via lm-evaluation-harness

✅ **Subjective Evaluation**: LLM judge assessment using server-hosted judge model

✅ **Results**: Aggregated to CSV for analysis

### Next Steps
- Run with `CONFIG = "full"` for comprehensive evaluation
- Compare results across different server configurations
- Extend to other MoE models available on the server

---

*This file was generated for MoE model validation - Feature 002-moe-qwen3-tests*

*Server-based adaptation for Lemonade Server OpenAI API compatibility*

*Modifications Copyright (c) 2025 AMD*